# Physical Bullying Detection from Video (RWF-2000 + VideoMAE)

This notebook trains a video classification model that predicts whether a short
video clip shows physical bullying / fighting or normal activity.

Dataset: RWF-2000 (Real World Fight 2000)
Model: VideoMAE (pretrained on Kinetics, fine-tuned here on RWF-2000)

The notebook is written as a step by step tutorial. Every section explains
what is happening and why, so you can follow the full pipeline from raw
video files to a trained model and predictions.

Pipeline:

Video -> Extract 16 Frames -> Resize 224x224 -> RGB -> VideoMAEImageProcessor
-> VideoMAE Model -> Classification Head -> Fight / NonFight

Run the cells from top to bottom. A GPU runtime is recommended
(Runtime -> Change runtime type -> GPU) but the notebook will also run on CPU,
just slower.

## Section 1 - Install Libraries

Install the packages this notebook needs. `transformers` gives us the VideoMAE model and processor. `kagglehub` is used later to download the dataset.

In [ ]:
# Install the required libraries.
# torch and torchvision usually already come pre-installed on Colab,
# but we install/upgrade them here so the notebook also works outside Colab.

!pip install -q torch torchvision
!pip install -q transformers
!pip install -q opencv-python-headless
!pip install -q scikit-learn
!pip install -q matplotlib
!pip install -q tqdm
!pip install -q kagglehub


## Section 2 - Import Libraries

All imports used in the notebook, grouped by purpose.

In [1]:
# Standard library
import os
import glob
import random

# Numerical / data handling
import numpy as np

# Video and image handling
import cv2

# Plotting
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hugging Face Transformers (VideoMAE model + processor)
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Progress bars
from tqdm.auto import tqdm

print("All libraries imported successfully.")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/home/s4lm-xi/anaconda3/lib/python3.9/runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/home/s4lm-xi/anaconda3/lib/python3.9/runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "/home/s4lm-xi/anaconda3/lib/python3.9/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/s4lm-xi/anaconda3/lib/python3.9/site-packages/traitlets/config/application.py", line 846, in launch_instance
    app.start()
  File "/h

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

## Section 3 - Configuration

All the settings used throughout the notebook are kept in one place so they
are easy to find and change. This cell also checks whether a GPU is
available and prints which device will be used for the rest of the
notebook. All later code re-uses this `DEVICE` variable, so training and
inference automatically run on GPU when one is available, and fall back to
CPU otherwise.

In [ ]:
# ----------------------------
# General settings
# ----------------------------
SEED = 42                 # random seed, for reproducibility
IMAGE_SIZE = 224           # frames are resized to IMAGE_SIZE x IMAGE_SIZE
NUM_FRAMES = 16             # number of frames sampled from each video
BATCH_SIZE = 4              # videos per training batch
NUM_EPOCHS = 5               # number of passes over the training set
LEARNING_RATE = 2e-5           # learning rate for the optimizer

# ----------------------------
# Label mapping
# ----------------------------
LABEL2ID = {"NonFight": 0, "Fight": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ----------------------------
# Reproducibility
# ----------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# GPU check
# ----------------------------
# This is the single place that decides whether we run on GPU or CPU.
# Every model, batch of data, and tensor we create later is moved to DEVICE,
# so the same code works whether or not a GPU is attached to this runtime.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU detected. Using device:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected. Using device: CPU (training will be slower).")


## Section 4 - Download / Locate the Dataset

The RWF-2000 dataset is distributed under a license that does not allow the
original authors to host the raw videos for open, automatic download. In
practice, most people get the videos in one of these ways:

1. A mirror of the dataset hosted on Kaggle (the cell below tries this
   automatically using `kagglehub`).
2. A manual download after being granted access by the dataset authors
   (https://github.com/mchengny/RWF2000-Video-Database-for-Violence-Detection),
   then uploading/unzipping it yourself (for example into Google Drive).

The notebook first tries the automatic Kaggle download. If that does not
work in your environment (for example, no Kaggle account/API key
configured), simply set `DATA_DIR` by hand in the second cell below to point
at a folder that already contains the `train/Fight`, `train/NonFight`,
`val/Fight`, `val/NonFight` structure, and skip the automatic download.

In [ ]:
# ----------------------------
# Option A: automatic download from Kaggle
# ----------------------------
# This uses kagglehub, which will ask you to log in / provide a Kaggle API
# token the first time it runs in a fresh Colab session.
# If you do not have a Kaggle account, skip this cell and use Option B below.

DATA_DIR = None  # will be set below if the automatic download succeeds

try:
    import kagglehub

    # This Kaggle dataset is a public mirror of RWF-2000 with the same
    # train/val, Fight/NonFight folder layout described in the assignment.
    downloaded_path = kagglehub.dataset_download("vulamnguyen/rwf2000")
    print("Downloaded dataset to:", downloaded_path)
    DATA_DIR = downloaded_path

except Exception as error:
    print("Automatic download did not complete.")
    print("Reason:", error)
    print("Please use Option B below and set DATA_DIR manually.")


In [ ]:
# ----------------------------
# Option B: manual dataset location
# ----------------------------
# If the automatic download above failed, or you already have the dataset
# somewhere (for example unzipped in Google Drive), set the path yourself
# here. This cell only overwrites DATA_DIR if you actually provide a path,
# so it is safe to run even after a successful automatic download.

MANUAL_DATA_DIR = ""  # <-- put your own path here if needed, e.g. "/content/drive/MyDrive/RWF-2000"

if MANUAL_DATA_DIR:
    DATA_DIR = MANUAL_DATA_DIR

if DATA_DIR is None:
    raise ValueError(
        "DATA_DIR is not set. Either fix the automatic download above, "
        "or set MANUAL_DATA_DIR to a folder containing the RWF-2000 data."
    )

print("Using dataset folder:", DATA_DIR)


In [ ]:
# ----------------------------
# Locate the actual train/val folders
# ----------------------------
# Some mirrors nest the data one level deeper (e.g. DATA_DIR/RWF-2000/train).
# This small helper looks for a folder that directly contains "train" and
# "val" subfolders, so the rest of the notebook does not need to worry
# about the exact folder depth of the download.

def find_dataset_root(start_path):
    for root, dirs, files in os.walk(start_path):
        if "train" in dirs and "val" in dirs:
            return root
    return start_path

DATASET_ROOT = find_dataset_root(DATA_DIR)
print("Dataset root folder:", DATASET_ROOT)
print("Contents:", os.listdir(DATASET_ROOT))


## Section 5 - Load Dataset

List every video file for the train and val splits, and print basic counts so we can sanity check the data before doing anything else.

In [ ]:
# Build a list of (video_path, label) pairs for a given split (train or val).

def list_videos(split_name):
    items = []
    for class_name, label in LABEL2ID.items():
        class_dir = os.path.join(DATASET_ROOT, split_name, class_name)
        video_paths = glob.glob(os.path.join(class_dir, "*.avi")) + \
                      glob.glob(os.path.join(class_dir, "*.mp4"))
        for video_path in video_paths:
            items.append((video_path, label))
    return items

train_items = list_videos("train")
val_items = list_videos("val")

train_fight_count = sum(1 for _, label in train_items if label == LABEL2ID["Fight"])
train_nonfight_count = sum(1 for _, label in train_items if label == LABEL2ID["NonFight"])
val_fight_count = sum(1 for _, label in val_items if label == LABEL2ID["Fight"])
val_nonfight_count = sum(1 for _, label in val_items if label == LABEL2ID["NonFight"])

print("Number of training videos:", len(train_items))
print("Number of validation videos:", len(val_items))
print("Training Fight videos:", train_fight_count)
print("Training NonFight videos:", train_nonfight_count)
print("Validation Fight videos:", val_fight_count)
print("Validation NonFight videos:", val_nonfight_count)


## Section 6 - Video Processing

This function turns a raw video file into a list of frames that the model
can use. We sample 16 frames evenly spaced across the video (instead of
using every frame) because this keeps the amount of data manageable while
still covering the full clip from start to end.

In [ ]:
def extract_frames(video_path, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
    """Open a video file and return a list of `num_frames` RGB frames.

    Steps:
    1. Open the video with OpenCV.
    2. Pick `num_frames` frame indices, evenly spaced across the video.
    3. For each chosen index, read the frame, convert it from BGR (OpenCV's
       default) to RGB, and resize it to (image_size, image_size).
    """
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Guard against corrupted videos that report 0 frames.
    if total_frames <= 0:
        total_frames = num_frames

    # Evenly spaced frame indices across the whole video.
    frame_indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    frames = []
    current_index = 0
    next_wanted = 0

    while cap.isOpened() and next_wanted < len(frame_indices):
        success, frame = cap.read()
        if not success:
            break

        if current_index == frame_indices[next_wanted]:
            # Convert color format: OpenCV reads frames as BGR, models expect RGB.
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize every frame to the same fixed size.
            frame = cv2.resize(frame, (image_size, image_size))
            frames.append(frame)
            next_wanted += 1

        current_index += 1

    cap.release()

    # If the video was shorter than expected, repeat the last frame
    # so we always return exactly `num_frames` frames.
    while len(frames) < num_frames:
        frames.append(frames[-1] if frames else np.zeros((image_size, image_size, 3), dtype=np.uint8))

    return frames


## Section 7 - Image Processing

`VideoMAEImageProcessor` handles the normalization steps the VideoMAE model
expects (scaling pixel values, mean/std normalization, etc). We only need to
hand it a list of RGB frames and it returns a ready-to-use tensor.

In [ ]:
# Load the processor that matches the pretrained VideoMAE checkpoint we will use.
MODEL_CHECKPOINT = "MCG-NJU/videomae-base-finetuned-kinetics"

image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)

def process_frames(frames):
    """Turn a list of RGB frames into a single pixel_values tensor
    shaped (num_frames, channels, height, width), ready for the model."""
    processed = image_processor(list(frames), return_tensors="pt")
    # processed["pixel_values"] has shape (1, num_frames, channels, height, width)
    # we drop the leading batch dimension here; the DataLoader adds it back later.
    return processed["pixel_values"][0]


## Section 8 - Dataset

A simple PyTorch `Dataset` that, for each video, extracts frames, processes them, and returns the resulting tensor together with its label.

In [ ]:
class BullyingVideoDataset(Dataset):
    def __init__(self, items):
        # items is a list of (video_path, label) tuples
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        video_path, label = self.items[index]

        # Step 1: read raw frames from the video file
        frames = extract_frames(video_path)

        # Step 2: turn frames into a normalized tensor the model expects
        video_tensor = process_frames(frames)

        return video_tensor, label

train_dataset = BullyingVideoDataset(train_items)
val_dataset = BullyingVideoDataset(val_items)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))


## Section 9 - DataLoaders

Wrap the datasets in `DataLoader`s so we can iterate over them in batches during training and validation.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))


## Section 10 - Visualize Data

Before training, let's look at one sample: the extracted frames and the
label they belong to. This helps confirm the preprocessing pipeline is
working as expected.

In [ ]:
# Pick one random training video to visualize.
sample_video_path, sample_label = random.choice(train_items)
sample_frames = extract_frames(sample_video_path)

print("Video file:", sample_video_path)
print("Label:", ID2LABEL[sample_label])

# Show a handful of the 16 extracted frames.
num_frames_to_show = 8
fig, axes = plt.subplots(1, num_frames_to_show, figsize=(16, 3))
step = len(sample_frames) // num_frames_to_show

for i in range(num_frames_to_show):
    axes[i].imshow(sample_frames[i * step])
    axes[i].axis("off")

fig.suptitle(f"Sample frames - Label: {ID2LABEL[sample_label]}")
plt.show()


## Section 11 - Load Model

We load a VideoMAE model that was already pretrained on Kinetics (a large
general action-recognition dataset), and replace its classification head
with a fresh 2-class head for our Fight / NonFight task. We do not freeze
any layers, so the whole network is fine-tuned on RWF-2000.

In [ ]:
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,  # allows replacing the old 400-class head with our 2-class head
)

# Move the model to GPU if available, otherwise it stays on CPU.
model.to(DEVICE)

print("Model loaded and moved to:", DEVICE)


## Section 12 - Training Loop

A straightforward training loop: for each epoch, we go through every
training batch, compute the loss, and update the model weights. Both the
model and every batch of data are sent to `DEVICE`, so this automatically
uses the GPU when one is available.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# These lists keep track of metrics after every epoch, used later for plotting.
train_losses = []
val_losses = []
val_accuracies = []

best_f1_score = 0.0

def run_training_epoch():
    """Runs one full pass over the training data and returns the average loss."""
    model.train()
    total_loss = 0.0

    for video_tensors, labels in tqdm(train_loader, desc="Training"):
        video_tensors = video_tensors.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(pixel_values=video_tensors)
        logits = outputs.logits

        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


## Section 13 - Validation

After every training epoch we evaluate the model on the validation set and
report loss, accuracy, precision, recall, and F1 score.

In [ ]:
def run_validation_epoch():
    """Runs one full pass over the validation data and returns
    (average loss, accuracy, precision, recall, f1)."""
    model.eval()
    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for video_tensors, labels in tqdm(val_loader, desc="Validation"):
            video_tensors = video_tensors.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(pixel_values=video_tensors)
            logits = outputs.logits

            loss = loss_function(logits, labels)
            total_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(val_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_predictions, average="binary", zero_division=0
    )

    return avg_loss, accuracy, precision, recall, f1, all_labels, all_predictions


## Section 14 - Run Training

This cell actually runs the training loop for `NUM_EPOCHS` epochs, printing metrics after each epoch and saving the best model (see Section 15).

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss = run_training_epoch()
    val_loss, val_accuracy, val_precision, val_recall, val_f1, _, _ = run_validation_epoch()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Train Loss:       {train_loss:.4f}")
    print(f"Validation Loss:  {val_loss:.4f}")
    print(f"Accuracy:         {val_accuracy:.4f}")
    print(f"Precision:        {val_precision:.4f}")
    print(f"Recall:           {val_recall:.4f}")
    print(f"F1 Score:         {val_f1:.4f}")

    # Section 15: Save Best Model
    # Only save the model weights when validation F1 improves,
    # so "best_model.pth" always holds the best checkpoint seen so far.
    if val_f1 > best_f1_score:
        best_f1_score = val_f1
        torch.save(model.state_dict(), "best_model.pth")
        print("New best model saved (F1 improved).")


## Section 15 - Save Best Model

Saving is already handled inside the training loop above: every time validation F1 improves, the model weights are written to `best_model.pth`. This cell just confirms the file exists and reports the best score reached.

In [ ]:
if os.path.exists("best_model.pth"):
    print("best_model.pth saved successfully.")
    print("Best validation F1 score reached:", round(best_f1_score, 4))
else:
    print("No checkpoint was saved yet. Run the training cell above first.")


## Section 16 - Training Graphs

Plot how loss and accuracy changed over the epochs.

In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, train_losses, label="Train Loss")
plt.plot(epochs_range, val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="green")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


## Section 17 - Confusion Matrix

Run the best saved model on the validation set once more and display a confusion matrix, to see exactly which classes are being confused.

In [ ]:
# Load the best checkpoint before evaluating, in case training kept going
# past the best epoch.
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.to(DEVICE)

_, _, _, _, _, final_labels, final_predictions = run_validation_epoch()

cm = confusion_matrix(final_labels, final_predictions)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")

class_names = [ID2LABEL[0], ID2LABEL[1]]
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.colorbar(im)
plt.show()


## Section 18 - Inference

A single, simple function that takes a video file path and returns the
predicted class and confidence. This reuses the exact same frame extraction
and processing steps used during training, so the model sees data in the
same format it was trained on.

In [ ]:
def predict(video_path):
    """Predict Fight / NonFight for a single video file.

    Steps: read video -> extract 16 frames -> resize -> process -> predict.
    """
    model.eval()

    frames = extract_frames(video_path)
    video_tensor = process_frames(frames)
    video_tensor = video_tensor.unsqueeze(0).to(DEVICE)  # add batch dimension

    with torch.no_grad():
        outputs = model(pixel_values=video_tensor)
        probabilities = torch.softmax(outputs.logits, dim=1)[0]
        predicted_id = int(torch.argmax(probabilities).item())
        confidence = float(probabilities[predicted_id].item())

    predicted_label = ID2LABEL[predicted_id]
    print(f"Prediction: {predicted_label}")
    print(f"Confidence: {confidence * 100:.1f}%")

    return predicted_label, confidence


## Section 19 - Test Example

Run the `predict` function on one validation video and compare the result against its actual label.

In [ ]:
# Pick one random validation video to test.
test_video_path, test_label = random.choice(val_items)
test_frames = extract_frames(test_video_path)

# Show a few frames from this test video.
num_frames_to_show = 8
fig, axes = plt.subplots(1, num_frames_to_show, figsize=(16, 3))
step = len(test_frames) // num_frames_to_show

for i in range(num_frames_to_show):
    axes[i].imshow(test_frames[i * step])
    axes[i].axis("off")

fig.suptitle("Test video frames")
plt.show()

predicted_label, confidence = predict(test_video_path)

print("Actual label:", ID2LABEL[test_label])
